# AI Travel Assistant Agent using LangChain, Groq, SerpAPI, Weather API, and Yagmail

This project demonstrates an AI-powered travel assistant agent that:
- searches travel destinations
- finds attractions and guides
- fetches weather information
- generates travel itineraries
- sends trip plans via email

Technologies Used:
- LangChain
- Groq LLM
- SerpAPI (Google Search)
- OpenWeather API
- Yagmail
- Python

In [ ]:
%pip install langchain


In [ ]:
%pip install langchain-groq

In [ ]:
%pip install google-search-results

In [ ]:
%pip install yagmail

In [ ]:
%pip install requests

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq

from serpapi import GoogleSearch

import yagmail
import requests

In [ ]:
GROQ_API_KEY = "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"

SERPAPI_API_KEY = "e5ca9a5494ffa2702da4621a19c3d5db4e96ad03ed220dbd771782fb856c9714"

WEATHER_API_KEY = "92dbdc780fd3226b6e96a112535bb69e"

In [ ]:
yag = yagmail.SMTP(
    "sairaashraf95134@gmail.com",
    "hxfy cmab jzwn dagh"
)

print("Email Service Connected!")

In [ ]:
@tool
def search_places_tool(query: str) -> str:
    """Search travel destinations, attractions, and travel guides."""

    params = {
        "engine": "google",
        "q": query,
        "api_key": SERPAPI_API_KEY,

        # IMPORTANT
        "num": 3
    }

    search = GoogleSearch(params)

    results = search.get_dict()

    # Return only small useful part
    if "organic_results" in results:
        simplified_results = []

        for r in results["organic_results"][:3]:
            simplified_results.append({
                "title": r.get("title"),
                "snippet": r.get("snippet"),
                "link": r.get("link")
            })

        return str(simplified_results)

    return "No results found."

In [ ]:
@tool
def weather_tool(city: str) -> str:
    """
    Get current weather information for a city.
    """

    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={WEATHER_API_KEY}&units=metric"

    response = requests.get(url)

    data = response.json()

    weather = data["weather"][0]["description"]
    temp = data["main"]["temp"]

    return f"Weather in {city}: {weather}, Temperature: {temp}°C"

In [ ]:
@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """
    Send travel itinerary email to recipient.
    """

    yag.send(
        to=recipient,
        subject=subject,
        contents=body
    )

    return "Travel itinerary email sent successfully!"

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

print("Groq Model Connected!")

In [ ]:
agent = create_agent(
    model=llm,

    tools=[
        search_places_tool,
        weather_tool,
        send_email_tool
    ],

    system_prompt="""
    You are an intelligent AI Travel Assistant.

    Your job is to:
    - search travel destinations
    - find attractions and travel guides
    - check weather conditions
    - generate travel itineraries
    - send trip plans through email

    Always provide helpful and organized travel plans.
    """
)

print("Travel Agent Created Successfully!")

In [ ]:
print(
    search_places_tool.invoke(
        "Best attractions in Murree"
    )
)

In [ ]:
print(
    weather_tool.invoke("Murree")
)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
Plan a 3-day trip to Murree.

Include:
- attractions
- weather
- activities

IMPORTANT:
After creating the itinerary,
USE the send_email_tool to send it to:

sairaashraf95134@gmail.com

Do NOT just say the email will be sent.
Actually call the tool.
"""
            }
        ]
    }
)

print(response["messages"][-1].content)